In [1]:
import pandas as pd

In [2]:
# Load filtered datasets
sold = pd.read_csv("filtered_residential_sold.csv", low_memory=False)
listings = pd.read_csv("filtered_residential_listings.csv", low_memory=False)

In [3]:
# Fetch the data
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])

In [4]:
# Lightly inspect the data
mortgage.columns = ['date', 'rate_30yr_fixed']
print("\nMortgage Data Preview:")
print(mortgage.head())


Mortgage Data Preview:
        date  rate_30yr_fixed
0 1971-04-02             7.33
1 1971-04-09             7.31
2 1971-04-16             7.31
3 1971-04-23             7.31
4 1971-04-30             7.29


In [5]:
# Resample to monthly
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)
print("\nMonthly Mortgage Rates:")
print(mortgage_monthly.head())


Monthly Mortgage Rates:
  year_month  rate_30yr_fixed
0    1971-04           7.3100
1    1971-05           7.4250
2    1971-06           7.5300
3    1971-07           7.6040
4    1971-08           7.6975


In [6]:
# Create join keys
# SOLD → use CloseDate
sold['CloseDate'] = pd.to_datetime(sold['CloseDate'], errors='coerce')
sold['year_month'] = sold['CloseDate'].dt.to_period('M')
# LISTINGS → use ListingContractDate
listings['ListingContractDate'] = pd.to_datetime(listings['ListingContractDate'], errors='coerce')
listings['year_month'] = listings['ListingContractDate'].dt.to_period('M')

In [7]:
# Merge
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

In [8]:
# Validate
print("\nMissing mortgage rates (sold):", sold_with_rates['rate_30yr_fixed'].isnull().sum())
print("Missing mortgage rates (listings):", listings_with_rates['rate_30yr_fixed'].isnull().sum())


Missing mortgage rates (sold): 0
Missing mortgage rates (listings): 0


In [9]:
# Preview
print("\nSold Preview with Rates:")
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head())


Sold Preview with Rates:
   CloseDate year_month  ClosePrice  rate_30yr_fixed
0 2024-01-26    2024-01    240000.0           6.6425
1 2024-01-05    2024-01    815000.0           6.6425
2 2024-01-05    2024-01    810000.0           6.6425
3 2024-01-30    2024-01    858000.0           6.6425
4 2024-01-29    2024-01   1890500.0           6.6425


In [10]:
# Write as CSV files
sold_with_rates.to_csv("sold_with_mortgage_rates.csv", index=False)
listings_with_rates.to_csv("listings_with_mortgage_rates.csv", index=False)